# Weights of evidence: from evidence layers to posterior odds

This lab follows the [method lesson](../methods/weights-of-evidence.md) and the [tested implementation](../implementations/python/weights-of-evidence/README.md). We will estimate weights from a small synthetic event map, inspect the resulting probability surface and see why duplicated evidence can create overconfidence.

The model combines binary evidence under conditional independence. The synthetic example is for learning, not a real prospectivity assessment.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

current = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (current, *current.parents)
     if (candidate / "implementations" / "python" / "weights-of-evidence").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
package_path = repo_root / "implementations" / "python" / "weights-of-evidence"
sys.path.insert(0, str(package_path))
from weights_of_evidence import weights_of_evidence


## 1. Build a synthetic study area

The event raster marks known occurrences. Two binary evidence themes represent a hypothetical fault-proximity zone and favourable lithology. In a real project, each theme needs a documented geological definition, spatial support and threshold.

In [ ]:
event = np.zeros((10, 10), dtype=int)
known_occurrences = [(1, 1), (1, 2), (2, 1), (2, 2), (3, 2), (4, 4),
                      (4, 5), (5, 4), (6, 7), (7, 7), (7, 8), (8, 7)]
for row, col in known_occurrences:
    event[row, col] = 1

fault_buffer = np.zeros_like(event)
fault_buffer[:, 1:5] = 1
favourable_lithology = np.zeros_like(event)
favourable_lithology[3:8, 3:8] = 1

evidence = {
    "fault_buffer": fault_buffer,
    "favourable_lithology": favourable_lithology,
}
result = weights_of_evidence(event, evidence, pseudocount=0.5)

print(f"Prior event prevalence: {result.prior_probability:.3f}")
for weight in result.weights:
    print(f"{weight.name}: W+={weight.positive:.3f}, "
          f"W-={weight.negative:.3f}, C={weight.contrast:.3f}")


## 2. Inspect the evidence and the result

The probability surface is conditional on the estimated weights, the prior event prevalence, the chosen binary themes and the model assumptions. It is not automatically a calibrated probability map.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
layers = [
    (event, "Known event cells"),
    (fault_buffer, "Fault-proximity evidence"),
    (favourable_lithology, "Favourable lithology"),
    (result.probability, "Model-based posterior probability"),
]
for ax, (layer, title) in zip(axes.flat, layers):
    im = ax.imshow(layer, cmap="viridis", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()


## 3. Inspect the contingency counts

Each evidence theme contributes a 2-by-2 table. The positive and negative weights are log ratios of conditional probabilities. The contrast describes the difference between the two weights.

In [ ]:
for weight in result.weights:
    print(f"\n{weight.name}")
    print("                 evidence=1  evidence=0")
    print(f"event=1          {weight.n_event_present:>9}  {weight.n_event_absent:>9}")
    print(f"event=0          {weight.n_non_event_present:>9}  {weight.n_non_event_absent:>9}")


## 4. Why conditional independence matters

Now duplicate one evidence theme under a different name. The implementation will estimate and add its contribution a second time. This is not new information: it is a deliberately invalid model that demonstrates how correlated or duplicated themes can inflate posterior odds.

In [ ]:
duplicated = {
    "fault_buffer": fault_buffer,
    "same_fault_signal_again": fault_buffer.copy(),
    "favourable_lithology": favourable_lithology,
}
duplicated_result = weights_of_evidence(event, duplicated, pseudocount=0.5)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].imshow(result.probability, cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Two distinct evidence themes")
axes[1].imshow(duplicated_result.probability, cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("One theme duplicated")
for ax in axes:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
plt.show()
print("Mean probability, original:", np.nanmean(result.probability))
print("Mean probability, duplicated:", np.nanmean(duplicated_result.probability))


## 5. Sensitivity to the pseudocount

A pseudocount prevents infinite weights when a contingency count is zero, but it changes the estimated probabilities. Compare a few declared values rather than treating smoothing as an invisible implementation detail.

In [ ]:
for alpha in (0.1, 0.5, 1.0, 2.0):
    fitted = weights_of_evidence(event, evidence, pseudocount=alpha)
    values = ", ".join(f"{w.name}: {w.positive:.3f}" for w in fitted.weights)
    print(f"alpha={alpha:>3}: {values}")


## 6. Questions to take away

1. Which evidence theme has the larger positive weight, and what do its counts say?
2. Why is the duplicated-theme model not a defensible improvement?
3. What could make an unrecorded occurrence different from a true absence?
4. How would you validate this map using spatially independent occurrences?
5. How would the results change if the evidence thresholds or study-area boundary changed?

Do not publish a prospectivity map until the occurrence data, evidence definitions, conditional-independence assumptions, sensitivity and validation design have been reviewed.